# Hack-A-Roo 2026, AI for Quantum track
# Notebook 8: Quantum graph embedding classifier

**Problem.** A graph has no natural vector representation, and most machine learning needs one. One way to get a vector is to encode the graph into a quantum circuit (one qubit per node, one entangling gate per edge), run it, and read out statistics of the measurement distribution. No training is involved in the circuit. The question is whether those statistics carry structure that a standard classical feature vector misses, and whether a downstream classifier can use them.

**What this notebook already does.** It generates graphs from three families with the same node count and the same edge count, so density cannot separate them. It groups the graphs into exact isomorphism classes and splits train and test by class, so no test graph is a relabeled copy of a training graph. It builds the encoding circuit gate by gate, computes the exact measurement distribution, and reads out a set of permutation-invariant statistics as the embedding. It computes a classical feature vector for the same graphs. It trains the same classifier on the quantum embedding, on each of its three blocks separately, on the classical features, and on both together, and reports held-out accuracy.

**Metric.** Held-out classification accuracy on unseen isomorphism classes, with balanced accuracy alongside it because a split by class does not keep the families balanced. Chance is one third.

**Two things the embedding is known to contain.** With the settings below, the single-qubit block is an exact function of the degree sequence (the circuit cell derives the formula and the notebook checks it), so that block cannot beat the classical features, which include the degree sequence. The odd Hamming weights are exactly zero by a conserved parity. The question that is open is whether the pairwise block carries anything the Laplacian spectrum, triangle count, clustering, and core numbers do not.

**Hypothesis to test.** On this task, the quantum embedding is competitive with the classical features, and combining them does not help much. The notebook does not assume this. The families are defined by their generators, not by a graph property, so a classifier can only learn what the generators leave behind in the graphs.

**Deliverable for the poster.** One figure (a bar chart of the accuracies with a confusion matrix for the quantum embedding) and the numbers. Then whichever shred-out you pick at the bottom.

Run every cell top to bottom once. The base run takes under a minute. Then go to the Shred-outs section.

In [ ]:
# CONFIG CELL. Every constant, seed, and path lives here. Nothing is hardcoded downstream.

RANDOM_SEED = 42

# Graphs. Every graph has NODE_COUNT nodes and EDGE_COUNT edges regardless of family.
# 12 nodes rather than 8: at 8 nodes there are only six 4-regular graphs, so the regular family is six topologies
# repeated and a class-grouped split leaves it almost nothing to test on. At 12 nodes there are 1544 connected ones.
NODE_COUNT = 12
EDGE_COUNT = 24
GRAPHS_PER_FAMILY = 300
# Families (the three classes):
#   "erdos_renyi"     uniformly random graph with exactly EDGE_COUNT edges
#   "watts_strogatz"  ring lattice with WATTS_STROGATZ_NEIGHBORS neighbors, each edge rewired with WATTS_STROGATZ_REWIRE_PROBABILITY
#   "random_regular"  every node has degree RANDOM_REGULAR_DEGREE
# The lattice and regular parameters must give EDGE_COUNT edges: NODE_COUNT * NEIGHBORS / 2 and NODE_COUNT * DEGREE / 2.
WATTS_STROGATZ_NEIGHBORS = 4
WATTS_STROGATZ_REWIRE_PROBABILITY = 0.3
RANDOM_REGULAR_DEGREE = 4

# Encoding circuit. See the circuit cell for exactly what each setting does.
EDGE_ANGLE = 1.0                # rzz angle on every edge
NODE_DEGREE_ANGLE = 0.0         # rz angle per unit of node degree; 0 turns the node term off
MIXER_GATE = "ry"               # "ry" or "rx", applied to every qubit before the Z measurement. See the circuit cell for why it matters.
MIXER_ANGLE = 1.5707963267948966    # pi/2. ry(pi/2) before a Z measurement is the same as measuring X.
ENCODING_LAYERS = 1

# Learning
TEST_FRACTION = 0.25            # fraction of ISOMORPHISM CLASSES held out
GRADIENT_BOOSTING_TREES = 300
GRADIENT_BOOSTING_DEPTH = 3

# Output
OUTPUT_PATH = "graph_embedding_dataset.pkl"
FIGURE_PATH = "graph_embedding_figure.png"

In [ ]:
# Install. Skip this cell if your environment already has these. Tested with qiskit 2.6, scikit-learn 1.8 and 1.9, networkx 3.7.
!pip install -q "qiskit[visualization]" "scikit-learn>=1.5" "networkx>=3.2" matplotlib

In [ ]:
import time
import pickle
import warnings
import numpy as np
import sklearn
import qiskit
import networkx as nx
import matplotlib.pyplot as plt
from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import confusion_matrix, balanced_accuracy_score

random_generator = np.random.default_rng(RANDOM_SEED)
FAMILY_NAMES = ["erdos_renyi", "watts_strogatz", "random_regular"]
assert NODE_COUNT * WATTS_STROGATZ_NEIGHBORS // 2 == EDGE_COUNT
assert NODE_COUNT * RANDOM_REGULAR_DEGREE // 2 == EDGE_COUNT
package_versions = {"numpy": np.__version__, "qiskit": qiskit.__version__, "scikit-learn": sklearn.__version__, "networkx": nx.__version__}
print("imports done", package_versions)

In [ ]:
# Generate the graphs. Each family gets its own seed stream from the shared generator.

def generate_graph(family):
    """One random graph from the named family with NODE_COUNT nodes and EDGE_COUNT edges"""
    seed = int(random_generator.integers(0, 2**31))
    if family == "erdos_renyi":
        return nx.gnm_random_graph(NODE_COUNT, EDGE_COUNT, seed=seed)
    if family == "watts_strogatz":
        return nx.watts_strogatz_graph(NODE_COUNT, WATTS_STROGATZ_NEIGHBORS, WATTS_STROGATZ_REWIRE_PROBABILITY, seed=seed)
    if family == "random_regular":
        return nx.random_regular_graph(RANDOM_REGULAR_DEGREE, NODE_COUNT, seed=seed)
    raise ValueError(f"unknown family {family}")


graphs = []
labels = []
for family_index, family in enumerate(FAMILY_NAMES):
    for graph_index in range(GRAPHS_PER_FAMILY):
        graph = generate_graph(family)
        assert graph.number_of_nodes() == NODE_COUNT
        assert graph.number_of_edges() == EDGE_COUNT
        graphs.append(graph)
        labels.append(family_index)
labels = np.array(labels)
print("generated", len(graphs), "graphs, all with", NODE_COUNT, "nodes and", EDGE_COUNT, "edges")
for family_index, family in enumerate(FAMILY_NAMES):
    example = graphs[family_index * GRAPHS_PER_FAMILY]
    print(f"  {family:16s} example degree sequence {sorted(dict(example.degree()).values())}, triangles {sum(nx.triangles(example).values()) // 3}")

In [ ]:
# Group the graphs by exact isomorphism class. A cheap invariant sorts graphs into buckets; an exact isomorphism test
# inside each bucket settles it. The invariant is the Weisfeiler-Lehman hash together with the rounded Laplacian
# spectrum: the hash alone puts every regular graph of the same degree in one bucket (1-WL cannot tell them apart),
# which made this cell take most of a minute on 300 regular graphs. Neither invariant is a proof of isomorphism,
# which is why the exact test is there.
#
# The label is the GENERATOR that produced a graph, not a property of the graph. One topology can in principle come
# out of two generators (a random graph that happens to be 4-regular, say). Such classes are counted and reported;
# a classifier cannot be blamed for them.

def isomorphism_class_ids(graph_list):
    """Assigns an integer class id to every graph so that two graphs share an id exactly when they are isomorphic"""
    hash_buckets = {}
    class_representatives = []
    class_ids = np.full(len(graph_list), -1)
    for graph_index, graph in enumerate(graph_list):
        spectrum = tuple(np.round(np.linalg.eigvalsh(nx.laplacian_matrix(graph).toarray()), 6))
        graph_hash = (nx.weisfeiler_lehman_graph_hash(graph), spectrum)
        for candidate_class_id in hash_buckets.get(graph_hash, []):
            if nx.is_isomorphic(graph, class_representatives[candidate_class_id]):
                class_ids[graph_index] = candidate_class_id
                break
        if class_ids[graph_index] == -1:
            new_class_id = len(class_representatives)
            class_representatives.append(graph)
            hash_buckets.setdefault(graph_hash, []).append(new_class_id)
            class_ids[graph_index] = new_class_id
    assert (class_ids >= 0).all()
    return class_ids


# networkx prints a one-time notice that its hash values changed in version 3.5. The hash is only a bucket key
# inside this one run, so the change is irrelevant here and the notice is silenced.
warnings.filterwarnings("ignore", message="The hashes produced for graphs without node or edge attributes changed")
start_time = time.time()
graph_class_ids = isomorphism_class_ids(graphs)
print(f"{len(np.unique(graph_class_ids))} isomorphism classes among {len(graphs)} graphs, found in {time.time() - start_time:.1f}s")
for family_index, family in enumerate(FAMILY_NAMES):
    print(f"  {family:16s} {len(np.unique(graph_class_ids[labels == family_index]))} distinct topologies among {GRAPHS_PER_FAMILY} graphs")
classes_with_several_labels = [class_id for class_id in np.unique(graph_class_ids) if len(np.unique(labels[graph_class_ids == class_id])) > 1]
print(f"  isomorphism classes that appear under more than one family label: {len(classes_with_several_labels)}")

In [ ]:
# The encoding circuit, built gate by gate. One qubit per node.
#
# For each layer:
#   1. H on every qubit (first layer only; later layers start from the previous state)
#   2. rz(NODE_DEGREE_ANGLE * degree) on each qubit                node term, off when the angle is 0
#   3. rzz(EDGE_ANGLE) on each edge                                 edge term, the graph structure
#   4. MIXER_GATE(MIXER_ANGLE) on every qubit                       mixer, makes the phases measurable
#
# Steps 2 and 3 are diagonal in the Z basis. Without the mixer in step 4, every Z-basis probability would be
# 1 / 2^n no matter what the graph is. The mixer turns the phases into populations.
#
# Which mixer matters. With the node term off, the state before the mixer is unchanged by flipping every qubit
# (X on all qubits fixes |+...+> and commutes with every rzz). An rx mixer commutes with that flip too, so the final
# distribution is symmetric under complementing every bit: every single-qubit <Z> is exactly zero and the
# Hamming-weight histogram is mirror-symmetric. Only the pairwise block carries information. An ry mixer does not
# commute with the flip (X ry(t) X = ry(-t)), so it breaks the symmetry and all three blocks are live. With
# MIXER_ANGLE = pi/2, ry before a Z measurement is the same as measuring X (with the sign flipped: ry(pi/2) sends
# |+> to |1>), and <X_i> on this state equals cos(EDGE_ANGLE) to the power of the degree of node i, so the measured
# <Z_i> is minus that. The single-qubit block is therefore a deterministic function of the degree sequence, and the
# classifier cell below checks what it is worth on its own. Try MIXER_GATE = "rx" to see the block vanish.
#
# One more conserved quantity with the node term off: the product of X over all qubits. |+...+> is its +1 eigenstate
# and every rzz commutes with it, so an X-basis measurement always returns an even number of -1 outcomes. After the
# ry(pi/2) mixer, a -1 outcome reads as bit 0, so the Hamming weight (number of 1 bits) is n minus an even number:
# for even n every odd weight is exactly zero, for odd n every even weight is. That is physics, not a bug.
#
# The embedding is permutation invariant, so relabeling the nodes gives the same vector. It is built from the
# exact probability vector:
#   a. Hamming-weight histogram: probability that exactly k qubits read 1, for k = 0 .. n           (n + 1 numbers)
#   b. single-qubit <Z> expectations, sorted                                                        (n numbers)
#   c. pairwise <Z_i Z_j> correlations over all pairs, sorted                                       (n(n-1)/2 numbers)

def build_encoding_circuit(graph):
    """Graph-to-circuit encoding described above"""
    circuit = QuantumCircuit(NODE_COUNT)
    circuit.h(range(NODE_COUNT))
    for layer_index in range(ENCODING_LAYERS):
        if NODE_DEGREE_ANGLE != 0:
            for node, degree in graph.degree():
                circuit.rz(NODE_DEGREE_ANGLE * degree, node)
        for node_i, node_j in graph.edges():
            circuit.rzz(EDGE_ANGLE, node_i, node_j)
        if MIXER_GATE == "ry":
            circuit.ry(MIXER_ANGLE, range(NODE_COUNT))
        elif MIXER_GATE == "rx":
            circuit.rx(MIXER_ANGLE, range(NODE_COUNT))
        else:
            raise ValueError(f"unknown MIXER_GATE {MIXER_GATE}")
    return circuit


# Bit table: bit_table[index, qubit] is the value of that qubit in the basis state with that statevector index.
# qiskit's index puts qubit 0 in the least significant bit.
state_indices = np.arange(2 ** NODE_COUNT)
bit_table = np.array([(state_indices >> qubit) & 1 for qubit in range(NODE_COUNT)]).T
z_sign_table = 1 - 2 * bit_table            # +1 for bit 0, -1 for bit 1
hamming_weights = bit_table.sum(axis=1)
qubit_pairs = [(qubit_i, qubit_j) for qubit_i in range(NODE_COUNT) for qubit_j in range(qubit_i + 1, NODE_COUNT)]
pair_sign_table = np.array([z_sign_table[:, qubit_i] * z_sign_table[:, qubit_j] for qubit_i, qubit_j in qubit_pairs]).T   # (states, pairs)

HISTOGRAM_SLICE = slice(0, NODE_COUNT + 1)
SINGLE_Z_SLICE = slice(NODE_COUNT + 1, 2 * NODE_COUNT + 1)
PAIR_ZZ_SLICE = slice(2 * NODE_COUNT + 1, 2 * NODE_COUNT + 1 + len(qubit_pairs))
EMBEDDING_NAMES = ([f"weight_{k}" for k in range(NODE_COUNT + 1)]
                   + [f"sorted_z_{k}" for k in range(NODE_COUNT)]
                   + [f"sorted_zz_{k}" for k in range(len(qubit_pairs))])


def quantum_embedding(graph):
    """Permutation-invariant readout statistics of the encoding circuit's exact measurement distribution"""
    probabilities = Statevector(build_encoding_circuit(graph)).probabilities()
    weight_histogram = np.array([probabilities[hamming_weights == k].sum() for k in range(NODE_COUNT + 1)])
    single_z = probabilities @ z_sign_table                 # (n,)
    pair_zz = probabilities @ pair_sign_table               # (pairs,)
    return np.concatenate([weight_histogram, np.sort(single_z), np.sort(pair_zz)])


example_graph = graphs[0]
print(build_encoding_circuit(example_graph).draw(fold=160))
example_embedding = quantum_embedding(example_graph)
assert example_embedding.shape == (len(EMBEDDING_NAMES),)
assert abs(example_embedding[HISTOGRAM_SLICE].sum() - 1) < 1e-9
print("embedding length:", len(example_embedding))
print("Hamming-weight histogram of the example:", np.round(example_embedding[HISTOGRAM_SLICE], 3))
print("sorted single-qubit <Z> of the example:  ", np.round(example_embedding[SINGLE_Z_SLICE], 3))

# Permutation-invariance check: relabel the example graph and compare embeddings.
permutation = random_generator.permutation(NODE_COUNT)
relabeled_graph = nx.relabel_nodes(example_graph, {node: int(permutation[node]) for node in example_graph.nodes()})
assert np.allclose(quantum_embedding(relabeled_graph), example_embedding)
print("embedding is invariant under node relabeling")

# The two closed forms stated above, checked when the settings they need are in force.
if MIXER_GATE == "ry" and NODE_DEGREE_ANGLE == 0 and ENCODING_LAYERS == 1 and abs(MIXER_ANGLE - np.pi / 2) < 1e-12:
    expected_single_z = np.sort([-np.cos(EDGE_ANGLE) ** degree for node, degree in example_graph.degree()])
    assert np.allclose(example_embedding[SINGLE_Z_SLICE], expected_single_z)
    print("single-qubit block matches the closed form -cos(EDGE_ANGLE)^degree, so it is a function of the degree sequence")
    vanishing_weights = np.arange(NODE_COUNT + 1)[(np.arange(NODE_COUNT + 1) % 2) != (NODE_COUNT % 2)]
    assert np.allclose(example_embedding[HISTOGRAM_SLICE][vanishing_weights], 0)
    print(f"Hamming weights {[int(weight) for weight in vanishing_weights]} are exactly zero, as the conserved X parity requires for n = {NODE_COUNT}")

In [ ]:
# Classical features for the same graphs: the standard things a graph-theory baseline would use.

CLASSICAL_NAMES = ([f"sorted_degree_{k}" for k in range(NODE_COUNT)]
                   + [f"laplacian_eigenvalue_{k}" for k in range(NODE_COUNT)]
                   + ["triangle_count", "average_clustering", "component_count", "max_core_number"])


def classical_features(graph):
    """Sorted degrees, Laplacian spectrum, triangles, clustering, components, degeneracy"""
    degrees = sorted(dict(graph.degree()).values())
    laplacian_eigenvalues = np.sort(np.linalg.eigvalsh(nx.laplacian_matrix(graph).toarray()))
    return np.concatenate([
        degrees,
        laplacian_eigenvalues,
        [sum(nx.triangles(graph).values()) / 3,
         nx.average_clustering(graph),
         nx.number_connected_components(graph),
         max(nx.core_number(graph).values())],
    ])


start_time = time.time()
quantum_rows = np.array([quantum_embedding(graph) for graph in graphs])
classical_rows = np.array([classical_features(graph) for graph in graphs])
assert quantum_rows.shape == (len(graphs), len(EMBEDDING_NAMES))
assert classical_rows.shape == (len(graphs), len(CLASSICAL_NAMES))
print(f"computed both feature sets for {len(graphs)} graphs in {time.time() - start_time:.1f}s")

In [ ]:
# Split by isomorphism class, then train the same classifier on six inputs and score on the same held-out graphs.
# The three quantum blocks are scored separately so the poster can say which block does the work.

group_splitter = GroupShuffleSplit(n_splits=1, test_size=TEST_FRACTION, random_state=RANDOM_SEED)
train_indices, test_indices = next(group_splitter.split(quantum_rows, labels, groups=graph_class_ids))
assert len(np.intersect1d(graph_class_ids[train_indices], graph_class_ids[test_indices])) == 0
print(f"{len(train_indices)} training graphs, {len(test_indices)} test graphs; no test graph is isomorphic to a training graph")
print("test graphs per family:", dict(zip(FAMILY_NAMES, np.bincount(labels[test_indices], minlength=len(FAMILY_NAMES)))))

feature_sets = {
    "quantum, all blocks": quantum_rows,
    "quantum, histogram only": quantum_rows[:, HISTOGRAM_SLICE],
    "quantum, single Z only": quantum_rows[:, SINGLE_Z_SLICE],
    "quantum, pair ZZ only": quantum_rows[:, PAIR_ZZ_SLICE],
    "classical features": classical_rows,
    "both": np.hstack([quantum_rows, classical_rows]),
}

results = {}
for set_name, rows in feature_sets.items():
    classifier = GradientBoostingClassifier(
        n_estimators=GRADIENT_BOOSTING_TREES, max_depth=GRADIENT_BOOSTING_DEPTH, random_state=RANDOM_SEED)
    classifier.fit(rows[train_indices], labels[train_indices])
    predictions = classifier.predict(rows[test_indices])
    results[set_name] = {
        "test_accuracy": (predictions == labels[test_indices]).mean(),
        "balanced_accuracy": balanced_accuracy_score(labels[test_indices], predictions),
        "confusion": confusion_matrix(labels[test_indices], predictions, labels=range(len(FAMILY_NAMES))),
        "importances": classifier.feature_importances_,
    }

print()
print(f"RESULTS ON {len(test_indices)} HELD-OUT GRAPHS FROM UNSEEN ISOMORPHISM CLASSES, chance = {1 / len(FAMILY_NAMES):.3f}")
print(f"{'features':28s}{'accuracy':>10s}{'balanced':>10s}")
for set_name in results:
    print(f"{set_name:28s}{results[set_name]['test_accuracy']:10.3f}{results[set_name]['balanced_accuracy']:10.3f}")
print(f"one test graph is {100 / len(test_indices):.1f} percentage points")
print()
print("top five quantum embedding statistics by importance (all blocks):")
quantum_importance_order = np.argsort(results["quantum, all blocks"]["importances"])[::-1]
for position in quantum_importance_order[:5]:
    print(f"  {EMBEDDING_NAMES[position]:16s} {results['quantum, all blocks']['importances'][position]:.3f}")
print("confusion matrix for the quantum embedding, all blocks (rows true, columns predicted, order", FAMILY_NAMES, "):")
print(results["quantum, all blocks"]["confusion"])

config = {
    "RANDOM_SEED": RANDOM_SEED,
    "NODE_COUNT": NODE_COUNT,
    "EDGE_COUNT": EDGE_COUNT,
    "GRAPHS_PER_FAMILY": GRAPHS_PER_FAMILY,
    "FAMILY_NAMES": FAMILY_NAMES,
    "WATTS_STROGATZ_NEIGHBORS": WATTS_STROGATZ_NEIGHBORS,
    "WATTS_STROGATZ_REWIRE_PROBABILITY": WATTS_STROGATZ_REWIRE_PROBABILITY,
    "RANDOM_REGULAR_DEGREE": RANDOM_REGULAR_DEGREE,
    "EDGE_ANGLE": EDGE_ANGLE,
    "NODE_DEGREE_ANGLE": NODE_DEGREE_ANGLE,
    "MIXER_GATE": MIXER_GATE,
    "MIXER_ANGLE": MIXER_ANGLE,
    "ENCODING_LAYERS": ENCODING_LAYERS,
    "TEST_FRACTION": TEST_FRACTION,
    "GRADIENT_BOOSTING_TREES": GRADIENT_BOOSTING_TREES,
    "GRADIENT_BOOSTING_DEPTH": GRADIENT_BOOSTING_DEPTH,
    "EMBEDDING_NAMES": EMBEDDING_NAMES,
    "CLASSICAL_NAMES": CLASSICAL_NAMES,
    "package_versions": package_versions,
}
dataset = {
    "config": config,
    "graphs": graphs,
    "labels": labels,
    "graph_class_ids": graph_class_ids,
    "quantum_rows": quantum_rows,
    "classical_rows": classical_rows,
    "train_indices": train_indices,
    "test_indices": test_indices,
    "results": results,
}
with open(OUTPUT_PATH, "wb") as output_file:
    pickle.dump(dataset, output_file)
print("saved", OUTPUT_PATH)

In [ ]:
# Poster figure: left, the accuracies. Right, the confusion matrix of the full quantum embedding.

figure, (left_axis, right_axis) = plt.subplots(1, 2, figsize=(13, 4.8))

set_names = list(results.keys())
accuracies = [results[set_name]["test_accuracy"] for set_name in set_names]
balanced = [results[set_name]["balanced_accuracy"] for set_name in set_names]
positions = np.arange(len(set_names))
left_axis.bar(positions - 0.2, accuracies, 0.4, label="accuracy")
left_axis.bar(positions + 0.2, balanced, 0.4, label="balanced accuracy")
left_axis.axhline(1 / len(FAMILY_NAMES), color="gray", linestyle="--", linewidth=0.8, label="chance")
left_axis.set_xticks(positions)
left_axis.set_xticklabels(set_names, rotation=25, ha="right", fontsize=8)
left_axis.set_ylim(0, 1)
left_axis.set_ylabel("held-out accuracy, unseen topologies")
left_axis.set_title("Which features classify the generator")
left_axis.legend(fontsize=8)

confusion = results["quantum, all blocks"]["confusion"]
right_axis.imshow(confusion, cmap="Blues")
right_axis.set_xticks(range(len(FAMILY_NAMES)))
right_axis.set_yticks(range(len(FAMILY_NAMES)))
right_axis.set_xticklabels(FAMILY_NAMES, rotation=30)
right_axis.set_yticklabels(FAMILY_NAMES)
right_axis.set_xlabel("predicted")
right_axis.set_ylabel("true")
right_axis.set_title("Quantum embedding (all blocks) confusion")
for row in range(len(FAMILY_NAMES)):
    for column in range(len(FAMILY_NAMES)):
        right_axis.text(column, row, confusion[row, column], ha="center", va="center")

figure.suptitle(f"{NODE_COUNT}-node, {EDGE_COUNT}-edge graphs, edge angle {EDGE_ANGLE}, mixer {MIXER_GATE}({MIXER_ANGLE:.2f}), {len(test_indices)} test graphs")
figure.tight_layout()
figure.savefig(FIGURE_PATH, dpi=200)
plt.show()
print("saved", FIGURE_PATH)

## Shred-outs

The base run answers one question on one task, and the block ablation says which part of the embedding did the work. A result here is a result against this classical baseline on this task, not against all classical graph representations. Pick one direction below, or invent your own. Each is written as the question your poster answers and the number that answers it. Always split by isomorphism class, report balanced accuracy, and choose settings on a validation split of classes before touching the test split.

1. **Change the task.** Replace the three families with a property to detect: is the graph bipartite, does it contain a triangle, is it planar, is it connected. Keep the edge count fixed. Which properties does the quantum embedding detect well, and which does it miss? Metric: held-out balanced accuracy per property, quantum versus classical, with the block ablation.

2. **Tune the encoding.** Sweep `EDGE_ANGLE` from 0.1 to 3.0, `MIXER_ANGLE` from 0.2 to pi/2, and `ENCODING_LAYERS` from 1 to 3. Switch `MIXER_GATE` to `"rx"` and watch the single-qubit block go to zero. Turn the node term on with `NODE_DEGREE_ANGLE`; the closed forms in the circuit cell stop holding, and the single-qubit block stops being a plain function of degree. Which settings make the embedding most informative? Metric: validation accuracy of each block versus each setting, and the variance of the embedding across graphs (an embedding that barely changes carries nothing).

3. **What does the embedding encode?** Regress from the quantum embedding to each classical feature in turn (triangle count, algebraic connectivity, clustering, max core number). The single-qubit block will recover the degree sequence exactly, by construction. Which other properties can be read off the pairwise block, and which cannot? Metric: held-out R² per property. The ones that cannot be read are either absent from the embedding or present in a form the regressor cannot unfold, and telling those apart is the real question.

4. **Hard pairs.** Find pairs of non-isomorphic graphs whose entire classical feature vector (as computed by `classical_features`, with an explicit numerical tolerance on the spectrum) is identical, by brute force over `networkx.graph_atlas_g()` (all graphs up to 7 nodes, so set `NODE_COUNT` to match). Equal degree sequence and equal spectrum alone are not enough: there are such pairs with different clustering, and clustering is in the vector. Verify each candidate pair against the actual feature function. Does the quantum embedding separate them? Metric: fraction of such pairs with embeddings that differ by more than numerical noise, and the minimum separation. A positive result is a result against this classical baseline, not against all of them.

5. **Scale the graphs.** Raise `NODE_COUNT` to 14 or 16 (keeping the regular degree and lattice neighbors consistent with `EDGE_COUNT`). The statevector grows to 65536 entries at 16 nodes, still fine. Does the embedding's accuracy hold, and does the number of pairwise statistics (n(n-1)/2) start to dominate? Metric: balanced accuracy versus node count for each feature set.

6. **Finite shots.** The base run uses the exact distribution. Sample it instead with `qiskit_aer` at 100, 1000, and 10000 shots and rebuild the embedding from counts. Every single-Z and pairwise-ZZ estimate is a mean of plus and minus ones with variance (1 minus its square) over the shots, so no block is intrinsically slower to converge; but there are n(n-1)/2 pairwise entries and they are sorted, so their total estimation error is larger. Metric: balanced accuracy versus shots, and which block degrades first.

7. **Use the embedding as a kernel.** Instead of statistics, use the overlap between two graphs' encoding states as a kernel and train an SVM (see notebook hackaroo26-3-quantum-kernel-svm for the kernel machinery). The overlap is not permutation invariant, so you will need to decide what to do about relabeling. Metric: balanced accuracy of the kernel SVM versus the embedding classifier.

8. **Downstream graph neural network.** Attach the quantum embedding as a graph-level feature to a small GNN trained on the same task. Does the GNN with the embedding beat the GNN without it? Metric: held-out balanced accuracy with and without, over five random class-grouped splits.

Whatever you choose, load `graph_embedding_dataset.pkl` for the base-run graphs, their isomorphism class ids, both feature sets, and the split. The `config` dict inside tells you exactly which run produced it.

In [ ]:
# Your extension starts here.
